# Channel Selection by Pearson Correlation

**Dataset**: MOABB BNCI2014-001 (Motor Imagery)  
**Channels**: 22 channels  
**Sampling rate**: 250 Hz  
**Subject**: 1

---

## Overview

We remove highly correlated channels (|r| > 0.85) to reduce redundancy.

## Expected outputs

- Correlation matrix with marks on removed channels
- Topomap showing kept vs removed channels

## Key parameters

| Parameter | Value |
| --- | --- |
| THRESHOLD | 0.85 |


## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB).


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


## 3. Explore the data


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


## 4. Compute correlation and remove redundancy


In [ ]:
THRESHOLD = 0.85

corr_sum = np.zeros((n_channels, n_channels))
for trial in range(n_trials):
    corr = np.corrcoef(X[trial, :, :])
    corr_sum += corr
corr_avg = corr_sum / n_trials

variances = np.var(X.reshape(n_trials, n_channels, n_samples), axis=(0, 2))
to_remove = set()
for i in range(n_channels):
    for j in range(i + 1, n_channels):
        if abs(corr_avg[i, j]) > THRESHOLD:
            if variances[i] < variances[j]:
                to_remove.add(i)
            else:
                to_remove.add(j)

kept = [i for i in range(n_channels) if i not in to_remove]
print(f'Kept {len(kept)} channels, removed {len(to_remove)}')


## 5. Interactive plot

**What to look for:**

- Adjacent channels are usually highly correlated
- We keep the channel with higher variance (more information)


In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=go.Heatmap(z=corr_avg, colorscale='RdBu_r', zmin=-1, zmax=1))
fig.update_layout(height=600, title='Channel Correlation Matrix', xaxis_title='Channel', yaxis_title='Channel')
fig.show()


## What did we learn?

- Removing redundant channels reduces dimensions without losing information
- The 0.85 threshold is not absolute, try different values
- A fast method that does not require model training
